In [1]:
# Install required acceleration and evaluation packages
!pip install -q transformers datasets accelerate nltk

import os
import zipfile
import urllib.request
import collections
import re
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
import torchvision.models as models
from PIL import Image
from transformers import AutoTokenizer, GPT2LMHeadModel
from nltk.translate.bleu_score import corpus_bleu, SmoothingFunction

# --- 1. Environment & Device Configuration ---
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using execution device: {device}")

# Download Flickr8k safely via public mirror URLs
DATA_URL = "https://github.com/jbrownlee/Datasets/releases/download/Flickr8k/Flickr8k_Dataset.zip"
TEXT_URL = "https://github.com/jbrownlee/Datasets/releases/download/Flickr8k/Flickr8k_text.zip"

if not os.path.exists("Flickr8k_Dataset.zip"):
    print("Downloading dataset images (approx 1GB)...")
    urllib.request.urlretrieve(DATA_URL, "Flickr8k_Dataset.zip")
if not os.path.exists("Flickr8k_text.zip"):
    print("Downloading text files...")
    urllib.request.urlretrieve(TEXT_URL, "Flickr8k_text.zip")

# Extract files
for zip_fn in ["Flickr8k_Dataset.zip", "Flickr8k_text.zip"]:
    with zipfile.ZipFile(zip_fn, 'r') as zip_ref:
        zip_ref.extractall(".")
print("Dataset extracted successfully.")


Using execution device: cuda
Dataset extracted successfully.


In [2]:
# --- 2. Dataset Splitting Logic (6000 Train, 1000 Val, 1000 Test) ---
def load_split_image_names(file_path):
    with open(file_path, 'r') as f:
        return set([line.strip() for line in f if line.strip()])

train_images_set = load_split_image_names("Flickr_8k.trainImages.txt")
val_images_set = load_split_image_names("Flickr_8k.devImages.txt")
test_images_set = load_split_image_names("Flickr_8k.testImages.txt")

def parse_captions(file_path):
    image_to_captions = collections.defaultdict(list)
    with open(file_path, 'r') as f:
        for line in f:
            tokens = line.strip().split('\t')
            if len(tokens) < 2:
                continue
            image_id, caption = tokens[0], tokens[1]
            image_name = image_id.split('#')[0]
            # Lowercase and strip punctuation
            caption_clean = caption.lower()
            caption_clean = re.sub(r'[^a-zA-Z\s]', '', caption_clean)
            image_to_captions[image_name].append(caption_clean.strip().split())
    return image_to_captions

all_captions = parse_captions("Flickr8k.token.txt")

# --- 3. Tokenizer Setup ---
tokenizer = AutoTokenizer.from_pretrained("gpt2")
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token


In [3]:
# --- 4. Custom PyTorch Dataset & Collate Fn ---
class FlickrResnetGptDataset(Dataset):
    def __init__(self, image_set, captions_dict, tokenizer, transform=None, img_dir="Flicker8k_Dataset"):
        self.image_names = list(image_set)
        self.captions_dict = captions_dict
        self.tokenizer = tokenizer
        self.transform = transform
        self.img_dir = img_dir

    def __len__(self):
        return len(self.image_names)

    def __getitem__(self, idx):
        img_name = self.image_names[idx]
        img_path = os.path.join(self.img_dir, img_name)
        image = Image.open(img_path).convert("RGB")

        if self.transform:
            image = self.transform(image)

        # Select first reference caption for training
        caption_words = self.captions_dict[img_name][0]
        caption_str = " ".join(caption_words)

        # Tokenize sentence and append closing End-Of-Text token
        tokens = self.tokenizer(caption_str, truncation=True, max_length=30, add_special_tokens=False)
        input_ids = tokens['input_ids'] + [self.tokenizer.eos_token_id]

        return image, torch.tensor(input_ids, dtype=torch.long)

def collate_fn(batch):
    images, captions = zip(*batch)
    images = torch.stack(images, 0)

    lengths = [len(cap) for cap in captions]
    max_len = max(lengths)

    # Pad inputs with GPT2's native EOS ID
    padded_captions = torch.full((len(captions), max_len), 50256, dtype=torch.long)
    attention_mask = torch.zeros((len(captions), max_len), dtype=torch.long)
    labels = torch.full((len(captions), max_len), -100, dtype=torch.long)

    for i, cap in enumerate(captions):
        padded_captions[i, :len(cap)] = cap
        attention_mask[i, :len(cap)] = 1
        labels[i, :len(cap)] = cap  # Target text for loss computation

    return images, padded_captions, attention_mask, labels

# Standard ResNet Normalization Pipeline
image_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

train_loader = DataLoader(FlickrResnetGptDataset(train_images_set, all_captions, tokenizer, image_transform),
                          batch_size=64, shuffle=True, collate_fn=collate_fn)

In [30]:
class ResNet50GPT2Captioner(nn.Module):
    def __init__(self, gpt2_model_name="gpt2"):
        super().__init__()
        # 1. ResNet50 Backbone Features Encoder
        resnet = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)
        # Freeze feature backbone to complete epochs in less time
        for param in resnet.parameters():
            param.requires_grad = False
        self.encoder = nn.Sequential(*list(resnet.children())[:-1]) # Output shape: (B, 2048, 1, 1)

        # 2. GPT2 Autoregressive Text Decoder
        self.decoder = GPT2LMHeadModel.from_pretrained(gpt2_model_name)

        # 3. Embedding Dimension Bridge Layer (2048 -> 768) with Dropout for regularization
        self.proj = nn.Linear(2048, self.decoder.config.n_embd)
        self.dropout = nn.Dropout(0.3) # Added dropout layer

    def forward(self, images, input_ids, attention_mask, labels):
        # Extract visual vectors securely
        with torch.no_grad():
            features = self.encoder(images)
        features = features.view(features.size(0), -1) # Flatten to (B, 2048)
        img_embeddings = self.dropout(self.proj(features)).unsqueeze(1) # Transform to (B, 1, 768) and apply dropout

        # Map text IDs to hidden embeddings
        text_embeddings = self.decoder.transformer.wte(input_ids) # (B, seq_len, 768)

        # Prepend the visual context vector to the front of the language vectors
        inputs_embeds = torch.cat((img_embeddings, text_embeddings), dim=1) # (B, seq_len + 1, 768)

        # Extend tracking mask to cover the new visual prompt position
        img_mask = torch.ones((attention_mask.size(0), 1), dtype=attention_mask.dtype, device=attention_mask.device)
        extended_attention_mask = torch.cat((img_mask, attention_mask), dim=1)

        # Pad loss target to ignore the image position via index cross-entropy flag -100
        dummy_labels = torch.full((labels.size(0), 1), -100, dtype=labels.dtype, device=labels.device)
        extended_labels = torch.cat((dummy_labels, labels), dim=1)

        outputs = self.decoder(inputs_embeds=inputs_embeds, attention_mask=extended_attention_mask, labels=extended_labels)
        return outputs

In [5]:
# --- 6. Fitting the Model Properly (Fast Training Pass) ---
model = ResNet50GPT2Captioner().to(device)

# Train only the projection layers and GPT-2 weights
optimizer = torch.optim.AdamW(list(model.proj.parameters()) + list(model.decoder.parameters()), lr=5e-4)
scaler = torch.amp.GradScaler('cuda')

print("\n--- Training ResNet50-GPT-2 Cross-Domain Alignment (1 Fast Epoch) ---")
model.train()
for step, (images, captions, masks, labels) in enumerate(train_loader):
    images, captions, masks, labels = images.to(device), captions.to(device), masks.to(device), labels.to(device)

    optimizer.zero_grad()
    with torch.amp.autocast('cuda'):
        outputs = model(images, captions, masks, labels)
        loss = outputs.loss

    scaler.scale(loss).backward()
    scaler.step(optimizer)
    scaler.update()

    if step % 20 == 0:
        print(f"Step [{step}/{len(train_loader)}], Cross-Entropy Loss: {loss.item():.4f}")

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



--- Training ResNet50-GPT-2 Cross-Domain Alignment (1 Fast Epoch) ---


`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Step [0/94], Cross-Entropy Loss: 6.0362
Step [20/94], Cross-Entropy Loss: 3.2542
Step [40/94], Cross-Entropy Loss: 3.0594
Step [60/94], Cross-Entropy Loss: 2.9703
Step [80/94], Cross-Entropy Loss: 2.8183


In [31]:
model = ResNet50GPT2Captioner().to(device)

# Train the projection layers and ALL GPT-2 weights
optimizer = torch.optim.AdamW(list(model.proj.parameters()) + list(model.decoder.parameters()), lr=5e-5) # Reduced learning rate for fine-tuning
scaler = torch.amp.GradScaler('cuda')

num_epochs = 10 # Changed number of epochs as per user request

print(f"\n--- Training ResNet50-GPT-2 Cross-Domain Alignment with GPT-2 Fine-tuning ({num_epochs} Epochs) ---")
for epoch in range(num_epochs):
    model.train()
    total_loss = 0
    for step, (images, captions, masks, labels) in enumerate(train_loader):
        images, captions, masks, labels = images.to(device), captions.to(device), masks.to(device), labels.to(device)

        optimizer.zero_grad()
        with torch.amp.autocast('cuda'):
            outputs = model(images, captions, masks, labels)
            loss = outputs.loss

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        total_loss += loss.item()

        if step % 20 == 0:
            print(f"Epoch [{epoch+1}/{num_epochs}], Step [{step}/{len(train_loader)}], Cross-Entropy Loss: {loss.item():.4f}")
    print(f"Epoch [{epoch+1}/{num_epochs}] Average Loss: {total_loss / len(train_loader):.4f}")

# Save the model after training
torch.save(model.state_dict(), "resnet_gpt2_captioner_fine_tuned.pth")
print("Model saved as resnet_gpt2_captioner_fine_tuned.pth")

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



--- Training ResNet50-GPT-2 Cross-Domain Alignment with GPT-2 Fine-tuning (10 Epochs) ---
Epoch [1/10], Step [0/94], Cross-Entropy Loss: 5.8196
Epoch [1/10], Step [20/94], Cross-Entropy Loss: 3.2569
Epoch [1/10], Step [40/94], Cross-Entropy Loss: 3.2080
Epoch [1/10], Step [60/94], Cross-Entropy Loss: 3.0120
Epoch [1/10], Step [80/94], Cross-Entropy Loss: 3.0346
Epoch [1/10] Average Loss: 3.3414
Epoch [2/10], Step [0/94], Cross-Entropy Loss: 2.8834
Epoch [2/10], Step [20/94], Cross-Entropy Loss: 2.6116
Epoch [2/10], Step [40/94], Cross-Entropy Loss: 2.8356
Epoch [2/10], Step [60/94], Cross-Entropy Loss: 2.8159
Epoch [2/10], Step [80/94], Cross-Entropy Loss: 2.6790
Epoch [2/10] Average Loss: 2.7662
Epoch [3/10], Step [0/94], Cross-Entropy Loss: 2.5576
Epoch [3/10], Step [20/94], Cross-Entropy Loss: 2.5525
Epoch [3/10], Step [40/94], Cross-Entropy Loss: 2.6395
Epoch [3/10], Step [60/94], Cross-Entropy Loss: 2.5959
Epoch [3/10], Step [80/94], Cross-Entropy Loss: 2.5864
Epoch [3/10] Averag

### Training for More Epochs

To improve the quality of the generated captions, the model needs to be trained for more epochs. A single epoch is typically not enough for complex tasks like image captioning to achieve good performance. Increasing the number of epochs will allow the model to learn more effectively from the data and refine its understanding of the image-caption relationships.

In [32]:
# --- 7. Advanced Autoregressive Beam Search Decoding ---
def beam_search_decode(model, image_tensor, tokenizer, beam_width=3, max_len=20):
    model.eval()
    with torch.no_grad():
        features = model.encoder(image_tensor.to(device))
        features = features.view(features.size(0), -1)
        img_embeds = model.proj(features).unsqueeze(1) # Shape: (1, 1, 768)

        start_id = tokenizer.eos_token_id
        candidates = [(0.0, [start_id])] # Format: (cumulative_log_probability, generated_token_ids)

        for _ in range(max_len):
            new_candidates = []
            for score, seq in candidates:
                if len(seq) > 1 and seq[-1] == tokenizer.eos_token_id:
                    new_candidates.append((score, seq))
                    continue

                text_ids = torch.tensor([seq], device=device)
                text_embeds = model.decoder.transformer.wte(text_ids)
                inputs_embeds = torch.cat((img_embeds, text_embeds), dim=1)

                outputs = model.decoder(inputs_embeds=inputs_embeds)
                logits = outputs.logits[:, -1, :]
                log_probs = torch.log_softmax(logits, dim=-1)

                top_probs, top_ids = log_probs.topk(beam_width, dim=-1)
                for i in range(beam_width):
                    new_candidates.append((score + top_probs[0][i].item(), seq + [top_ids[0][i].item()]))

            candidates = sorted(new_candidates, key=lambda x: x[0], reverse=True)[:beam_width]
            if all(len(seq) > 1 and seq[-1] == tokenizer.eos_token_id for _, seq in candidates):
                break

        best_seq = candidates[0][1]
        return tokenizer.decode(best_seq, skip_special_tokens=True).lower().split()

In [24]:
# --- 8. Validation Evaluation via BLEU Metric ---
print("\n--- Evaluating Hybrid Architecture on Validation Split ---")
val_dataset = FlickrResnetGptDataset(val_images_set, all_captions, tokenizer, image_transform)

references = []
hypotheses = []

# Process a subset of 50 samples for near-instant metric verification
for idx in range(50):
    img_name = val_dataset.image_names[idx]
    img_tensor, _ = val_dataset[idx]

    predicted_words = beam_search_decode(model, img_tensor.unsqueeze(0), tokenizer, beam_width=3)

    references.append(all_captions[img_name])
    hypotheses.append(predicted_words)

smooth = SmoothingFunction().method1
bleu1 = corpus_bleu(references, hypotheses, weights=(1.0, 0, 0, 0), smoothing_function=smooth)
bleu4 = corpus_bleu(references, hypotheses, weights=(0.25, 0.25, 0.25, 0.25), smoothing_function=smooth)

print(f"\nResNet50 + GPT-2 Validation Metrics:")
print(f"BLEU-1 Score (Single Token Match): {bleu1:.4f}")
print(f"BLEU-4 Score (Phrase Fluidity Match): {bleu4:.4f}")


--- Evaluating Hybrid Architecture on Validation Split ---

ResNet50 + GPT-2 Validation Metrics:
BLEU-1 Score (Single Token Match): 0.4178
BLEU-4 Score (Phrase Fluidity Match): 0.0659


### Full Validation Evaluation

Previously, the BLEU score was calculated on a subset of 50 samples. To get a more accurate and representative evaluation of the model's performance, we will now calculate the BLEU scores on the entire validation dataset.

In [33]:
# --- 8. Validation Evaluation via BLEU Metric (Full Dataset) ---
print("\n--- Evaluating Hybrid Architecture on Full Validation Split ---")
val_dataset = FlickrResnetGptDataset(val_images_set, all_captions, tokenizer, image_transform)

references = []
hypotheses = []

# Process the entire validation set
for idx in range(len(val_dataset)):
    img_name = val_dataset.image_names[idx]
    img_tensor, _ = val_dataset[idx]

    predicted_words = beam_search_decode(model, img_tensor.unsqueeze(0), tokenizer, beam_width=3)

    references.append(all_captions[img_name])
    hypotheses.append(predicted_words)

smooth = SmoothingFunction().method1
bleu1 = corpus_bleu(references, hypotheses, weights=(1.0, 0, 0, 0), smoothing_function=smooth)
bleu4 = corpus_bleu(references, hypotheses, weights=(0.25, 0.25, 0.25, 0.25), smoothing_function=smooth)

print(f"\nResNet50 + GPT-2 Validation Metrics (Full Dataset):")
print(f"BLEU-1 Score (Single Token Match): {bleu1:.4f}")
print(f"BLEU-4 Score (Phrase Fluidity Match): {bleu4:.4f}")



--- Evaluating Hybrid Architecture on Full Validation Split ---

ResNet50 + GPT-2 Validation Metrics (Full Dataset):
BLEU-1 Score (Single Token Match): 0.5612
BLEU-4 Score (Phrase Fluidity Match): 0.1528


In [34]:
# --- 9. Sample Test Prediction ---
print("\n--- Generating Evaluation Sample Inference From Test Split ---")
test_dataset = FlickrResnetGptDataset(test_images_set, all_captions, tokenizer, image_transform)
sample_img_tensor, _ = test_dataset[12]
sample_caption = beam_search_decode(model, sample_img_tensor.unsqueeze(0), tokenizer, beam_width=3)
print(f"Predicted Test Caption: {' '.join(sample_caption)}")


--- Generating Evaluation Sample Inference From Test Split ---
Predicted Test Caption: a group of children playing in a field


In [35]:
import matplotlib.pyplot as plt

print("\n--- Visualizing Sample Test Predictions (Condensed) ---")

# Display 5 sample images with their true and predicted captions
num_samples_to_display = 5

for i in range(num_samples_to_display):
    # Get a random image from the test set
    sample_idx = torch.randint(0, len(test_dataset), (1,)).item()
    img_name = test_dataset.image_names[sample_idx]
    img_tensor, _ = test_dataset[sample_idx]

    # Get the original image for display (without normalization)
    original_img_path = os.path.join(test_dataset.img_dir, img_name)
    original_image = Image.open(original_img_path).convert("RGB")

    # Generate prediction
    predicted_words = beam_search_decode(model, img_tensor.unsqueeze(0), tokenizer, beam_width=3)
    predicted_caption = ' '.join(predicted_words)

    # Get true captions and format them for condensed display
    true_captions_list = [" ".join(cap) for cap in all_captions[img_name]]
    true_captions_formatted = "\n".join([f"- {cap}" for cap in true_captions_list])

    # Create a figure with two subplots, with a smaller figure size
    fig, axes = plt.subplots(1, 2, figsize=(12, 5)) # Reduced figsize

    # Plot image on the left subplot
    axes[0].imshow(original_image)
    axes[0].axis('off')
    axes[0].set_title(f"Image: {img_name}", fontsize=10) # Smaller title font

    # Display captions on the right subplot in a more condensed way
    axes[1].axis('off')
    axes[1].text(0, 1.0, "True Captions:", fontsize=12, verticalalignment='top') # Smaller font
    axes[1].text(0, 0.95, true_captions_formatted, fontsize=10, verticalalignment='top', wrap=True) # Smaller font, consolidated
    axes[1].text(0, 0.4, "Predicted Caption:", fontsize=12, verticalalignment='top') # Smaller font
    axes[1].text(0, 0.35, predicted_caption, fontsize=10, verticalalignment='top', wrap=True) # Smaller font
    axes[1].set_xlim(0, 1)
    axes[1].set_ylim(0, 1)

    plt.tight_layout() # Adjust layout to prevent labels from being cut off
    plt.show()

Output hidden; open in https://colab.research.google.com to view.

In [39]:
import nbformat
import os
from google.colab import drive

# --- IMPORTANT: Configure this path correctly ---
# 1. Mount Google Drive
drive.mount('/content/drive')

# 2. Set notebook_path to the FULL path of your notebook in Google Drive.
#    Example: '/content/drive/MyDrive/MyFolder/my_image_captioning_notebook.ipynb'
#    If the notebook is in the root of MyDrive, it would be '/content/drive/MyDrive/image.ipynb'
#    Based on the error, it seems 'image.ipynb' is NOT directly in MyDrive, or the name is slightly different.
#    You can find the path by navigating to your notebook in Colab's file browser (left sidebar),
#    right-clicking on it, and selecting 'Copy path'.
notebook_path = '/content/drive/MyDrive/image.ipynb' # <--- CRITICAL: VERIFY/ADJUST THIS PATH!
output_notebook_path = '/content/Notebook_for_GitHub.ipynb' # This will be saved locally in Colab

# Initialize notebook to None
notebook = None

# Load the notebook
try:
    with open(notebook_path, 'r', encoding='utf-8') as f:
        notebook = nbformat.read(f, as_version=4)
except FileNotFoundError:
    print(f"\nError: Notebook not found at {notebook_path}.")
    print("Please verify that 'notebook_path' is set correctly to your notebook's exact FULL path in Google Drive.")
    print("You can find the path by navigating to your notebook in Colab's file browser (left sidebar), right-clicking it, and selecting 'Copy path'.")
except Exception as e:
    print(f"\nAn unexpected error occurred while loading the notebook: {e}")

# Only proceed if the notebook was successfully loaded
if notebook:
    # Remove the 'widgets' entry from the metadata if it exists
    if 'widgets' in notebook.metadata:
        del notebook.metadata['widgets']
        print("\nRemoved 'widgets' metadata.")
    else:
        print("\nNo 'widgets' metadata found to remove.")

    # Save the modified notebook to a new file in the Colab environment
    with open(output_notebook_path, 'w', encoding='utf-8') as f:
        nbformat.write(notebook, f)

    print(f"\nModified notebook saved to: {output_notebook_path}")
    print("\n--- Next Steps ---")
    print("1. Look for the file 'Notebook_for_GitHub.ipynb' in the Colab file browser (the folder icon on the left sidebar).")
    print("2. Right-click on 'Notebook_for_GitHub.ipynb' and select 'Download'.")
    print("3. Go to your GitHub repository and manually upload this downloaded file.")
else:
    print("\nNotebook was not loaded, unable to process for GitHub upload.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

Removed 'widgets' metadata.

Modified notebook saved to: /content/Notebook_for_GitHub.ipynb

--- Next Steps ---
1. Look for the file 'Notebook_for_GitHub.ipynb' in the Colab file browser (the folder icon on the left sidebar).
2. Right-click on 'Notebook_for_GitHub.ipynb' and select 'Download'.
3. Go to your GitHub repository and manually upload this downloaded file.


### Exploring Beam Search Width

Beam search is a heuristic search algorithm used to expand the most promising nodes in a search tree. In the context of sequence generation (like captioning), `beam_width` (often denoted as `k`) determines how many of the most likely sequences are kept at each step of the decoding process.

*   **Higher `beam_width`**: Generally leads to more diverse and potentially better quality captions by exploring a wider range of possibilities. However, it also increases computational cost and can sometimes produce grammatically correct but less relevant captions.
*   **Lower `beam_width`**: Is faster but might miss out on better caption sequences due to its limited exploration.